# 14 — PCSE-WOFOST benchmark for HRW Wheat

**Purpose**: Run WOFOST (Wageningen process-based crop model) for ~24,500 field-years and extract phenology dates as an **independent benchmark** for the paper.

## Why WOFOST is meaningful here
- **Independent of our WES framework** — different physics (TSUM-based, not Wang-Engel)
- **Widely used** by JRC MARS, Wageningen for crop monitoring
- **Phenology**: predicts DVS=0 (emergence), DVS=1 (anthesis), DVS=2 (maturity)

## Limitations (paper-honest)
- WOFOST does NOT have separate tillering/jointing/flag-leaf/boot/heading stages
- Default Winter_wheat_106 from European conditions; US Plains differ
- Daymet weather only from 2013-01-01 → harvest_year=2013 fields (sown fall 2012) are excluded
- Default Oct 15 sowing of harvest_year-1
- Daymet srad converted to daily total via astronomical daylength
- Wind set to 3 m/s (US Plains avg, Daymet has no wind)

## Output
`pcse_wofost_phenology.csv` — per (field_id, year): wofost_emergence_doy, wofost_anthesis_doy, wofost_maturity_doy

In [ ]:
import pandas as pd
import numpy as np
import os, sys, datetime as dt
import warnings
warnings.filterwarnings('ignore')

from pcse.input import YAMLCropDataProvider, WOFOST72SiteDataProvider, DummySoilDataProvider
from pcse.base import WeatherDataProvider, ParameterProvider, WeatherDataContainer
from pcse.models import Wofost72_PP
from pcse.util import reference_ET, daylength
import yaml as yamlpkg

# Paths
FEAT_PATH    = 'data/processed/features/features_with_external.parquet'
WEATHER_PATH = 'data/raw/satellite/daymet_daily_weather_full.csv'
OUT_PATH     = 'data/results/pcse_wofost_phenology.csv'

feat = pd.read_parquet(FEAT_PATH)
feat['field_id'] = feat['field_id'].astype(str)
feat['year'] = feat['year'].astype(int)
# Daymet starts 2013-01-01 → harvest_year=2013 has no fall 2012 sowing weather
feat = feat[feat['year'] >= 2014].reset_index(drop=True)
print(f'Features (2014-2017): {feat.shape}')

## 1. DaymetWeatherProvider

In [ ]:
class DaymetWeatherProvider(WeatherDataProvider):
    """Wraps a per-field Daymet DataFrame for PCSE.
    Daymet provides Tmin, Tmax (°C), srad (W/m² daylight-avg), vp (Pa), prcp (mm).
    PCSE wants Tmin/Tmax (°C), IRRAD (J/m²/day), VAP (hPa), WIND (m/s),
    RAIN (cm/day), ET0/ES0/E0 (cm/day).
    Daymet srad must be multiplied by daylight seconds (not 86400) to get daily total.
    """
    def __init__(self, df, latitude, longitude, elevation=400):
        super().__init__()
        self.latitude = latitude
        self.longitude = longitude
        self.elevation = elevation
        self.angstA = 0.18
        self.angstB = 0.55
        df = df.sort_values('date').reset_index(drop=True)
        for _, row in df.iterrows():
            d = row['date'].date() if hasattr(row['date'], 'date') else row['date']
            tmin = float(row['Tmin'])
            tmax = float(row['Tmax'])
            dayl_h = daylength(d, self.latitude)
            irrad = float(row['srad']) * dayl_h * 3600.0
            vap_kpa = float(row['vp']) / 1000.0
            rain_cm = float(row['prcp']) / 10.0
            wind = 3.0
            try:
                e0, es0, et0 = reference_ET(d, self.latitude, self.elevation,
                                            tmin, tmax, irrad, vap_kpa*10.0,
                                            wind, self.angstA, self.angstB)
                e0  = min(max(e0,  0.0), 2.5)
                es0 = min(max(es0, 0.0), 2.5)
                et0 = min(max(et0, 0.0), 2.5)
            except Exception:
                e0 = es0 = et0 = 0.5
            wdc = WeatherDataContainer(
                LAT=self.latitude, LON=self.longitude, ELEV=self.elevation,
                DAY=d,
                IRRAD=irrad, TMIN=tmin, TMAX=tmax,
                VAP=vap_kpa*10.0, WIND=wind, RAIN=rain_cm,
                E0=e0, ES0=es0, ET0=et0, SNOWDEPTH=0.0,
            )
            self._store_WeatherDataContainer(wdc, d)

print('DaymetWeatherProvider defined.')

## 2. Load weather and pre-group by field

In [ ]:
fields_of_interest = set(feat['field_id'].unique())
print(f'Fields to load weather for: {len(fields_of_interest)}')

weather_chunks = []
for chunk in pd.read_csv(WEATHER_PATH, chunksize=500_000):
    chunk['FIELDID'] = chunk['FIELDID'].astype(str)
    chunk = chunk[chunk['FIELDID'].isin(fields_of_interest)]
    if len(chunk) > 0:
        weather_chunks.append(chunk)
wx = pd.concat(weather_chunks, ignore_index=True)
wx['date'] = pd.to_datetime(wx['date'])
wx_by_field = {fid: g.sort_values('date').reset_index(drop=True) for fid, g in wx.groupby('FIELDID')}
print(f'Weather loaded: {len(wx_by_field)} fields, {len(wx):,} rows')

## 3. Crop / soil / site setup

In [ ]:
crop = YAMLCropDataProvider()
crop.set_active_crop('wheat', 'Winter_wheat_106')
soil = DummySoilDataProvider()
site = WOFOST72SiteDataProvider(WAV=10)
params = ParameterProvider(soildata=soil, cropdata=crop, sitedata=site)
print(f'TSUM1 (emergence→anthesis) = {crop["TSUM1"]:.0f} °C·d')
print(f'TSUM2 (anthesis→maturity)  = {crop["TSUM2"]:.0f} °C·d')

## 4. Simulation function

In [ ]:
def build_agromgmt(sow_date, end_date):
    campaign_start = sow_date - dt.timedelta(days=7)
    yaml_str = f"""
- {campaign_start}:
    CropCalendar:
        crop_name: wheat
        variety_name: Winter_wheat_106
        crop_start_date: {sow_date}
        crop_start_type: sowing
        crop_end_date: {end_date}
        crop_end_type: maturity
        max_duration: 350
    TimedEvents: null
    StateEvents: null
"""
    return yamlpkg.safe_load(yaml_str)

def extract_phenology(out_df):
    """DOY (calendar) where DVS crosses thresholds."""
    res = {}
    for stage_name, dvs_thresh in [('emergence', 0.0), ('anthesis', 1.0), ('maturity', 2.0)]:
        if stage_name == 'emergence':
            mask = out_df['DVS'].notna() & (out_df['DVS'] > 0)
        else:
            mask = out_df['DVS'] >= dvs_thresh
        if not mask.any():
            res[stage_name] = np.nan
            continue
        first_day = out_df.loc[mask.idxmax(), 'day']
        res[stage_name] = first_day.timetuple().tm_yday
    return res

def simulate_one(fid, yr, lat, lon, wx_field):
    """Run WOFOST for a single field-year. Default sowing Oct 15 of yr-1."""
    try:
        wx_gs = wx_field[(wx_field['date'] >= pd.Timestamp(f'{yr-1}-07-01')) &
                         (wx_field['date'] <= pd.Timestamp(f'{yr}-07-15'))]
        if len(wx_gs) < 200:
            return {'field_id': fid, 'year': yr, 'error': 'insufficient_weather'}
        wp = DaymetWeatherProvider(wx_gs, latitude=lat, longitude=lon, elevation=400)
        sow_date = dt.date(yr-1, 10, 15)
        end_date = dt.date(yr, 7, 15)
        agro = build_agromgmt(sow_date, end_date)
        wofost = Wofost72_PP(params, wp, agro)
        wofost.run_till_terminate()
        out = pd.DataFrame(wofost.get_output())
        out['day'] = pd.to_datetime(out['day'])
        ph = extract_phenology(out)
        return {'field_id': fid, 'year': yr,
                'wofost_emergence_doy': ph['emergence'],
                'wofost_anthesis_doy': ph['anthesis'],
                'wofost_maturity_doy': ph['maturity']}
    except Exception as e:
        return {'field_id': fid, 'year': yr, 'error': str(e)[:120]}

# PoC: single field-year
row = feat.iloc[0]
fid, yr, lat, lon = row['field_id'], int(row['year']), float(row['latitude']), float(row['longitude'])
if fid in wx_by_field:
    poc = simulate_one(fid, yr, lat, lon, wx_by_field[fid])
    print(f'PoC: {poc}')

## 5. Batch — sample run (100 fields)

In [ ]:
import time

SAMPLE_N = 100
feat_sample = feat.sample(n=min(SAMPLE_N, len(feat)), random_state=42)
results = []
t0 = time.time()
for i, (_, r) in enumerate(feat_sample.iterrows()):
    fid = r['field_id']; yr = int(r['year'])
    if fid not in wx_by_field: continue
    res = simulate_one(fid, yr, r['latitude'], r['longitude'], wx_by_field[fid])
    results.append(res)
    if (i+1) % 20 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/{SAMPLE_N} done in {elapsed:.0f}s ({elapsed/(i+1):.2f}s/sim)')

df_sample = pd.DataFrame(results)
print(f'\nSample done: {len(df_sample)} sims, total time {time.time()-t0:.0f}s')
print(df_sample.head(10))
if 'wofost_anthesis_doy' in df_sample.columns:
    print(f'\nAnthesis DOY: median={df_sample["wofost_anthesis_doy"].median():.0f}')
    print(f'Maturity DOY: median={df_sample["wofost_maturity_doy"].median():.0f}')
if 'error' in df_sample.columns:
    err_count = df_sample['error'].notna().sum()
    print(f'Errors: {err_count}')
    if err_count > 0:
        print(df_sample[df_sample['error'].notna()]['error'].value_counts().head(5))

## 6. Full batch (parallel) — uncomment to run

In [ ]:
RUN_FULL = True
if RUN_FULL:
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing as mp
    # Note: ProcessPoolExecutor needs `simulate_one` to be picklable.
    # If issues, fall back to sequential or joblib.Parallel.
    all_results = []
    n_workers = max(1, mp.cpu_count() - 2)
    jobs = [(r['field_id'], int(r['year']), r['latitude'], r['longitude'])
            for _, r in feat.iterrows() if r['field_id'] in wx_by_field]
    def worker(args):
        fid, yr, lat, lon = args
        return simulate_one(fid, yr, lat, lon, wx_by_field[fid])
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = [ex.submit(worker, j) for j in jobs]
        for i, fut in enumerate(as_completed(futures)):
            all_results.append(fut.result())
            if (i+1) % 500 == 0: print(f'  {i+1}/{len(jobs)} done')
    df_full = pd.DataFrame(all_results)
    df_full.to_csv(OUT_PATH, index=False)
    print(f'Saved: {OUT_PATH}, shape: {df_full.shape}')